In [23]:
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents
from minsearch import Index
from openai import OpenAI

In [17]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

In [24]:
files = reader.read()

In [25]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [26]:
chunks = chunk_documents(documents, size=2000, step=1000)


In [27]:
index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
index.fit(chunks)


In [28]:
type(index)

minsearch.minsearch.Index

In [29]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback


agent_tools = Tools()


In [ ]:
from minsearch import Index

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    #boost_dict = {"question": 3.0, "section": 0.5}
    #filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5
    )

In [42]:
agent_tools.add_tool(search)

In [43]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'},
    'index': {'type': 'string', 'description': 'index parameter'}},
   'required': ['query', 'index'],
   'additionalProperties': False}}]

In [33]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [34]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""

In [35]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [36]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received
